# DocuMind — affinage GPU du modèle de recherche (embeddings)

Ce notebook réentraîne `intfloat/multilingual-e5-small` sur les mêmes données que le projet, mais sur GPU :
3 époques complètes, lots de 64, séquences de 256 jetons (la version CPU n'avait fait que 300 pas, soit 0,43 époque).

**Avant de lancer (panneau de droite de Kaggle) :** *Accelerator* = **GPU T4** et *Internet* = **On**.
Puis **Run All** (ou *Save Version → Save & Run All*). Durée attendue : 10 à 20 minutes.

**À la fin, récupérer dans l'onglet Output :**
1. `documind-e5-small-ft.zip` (le modèle) ;
2. `retrieval_benchmark.json` (les mesures) ;
3. ce notebook avec ses sorties (*File → Download notebook*).

In [ ]:
!nvidia-smi -L
!pip install -q -U "sentence-transformers>=5.0" datasets accelerate rank-bm25
import sentence_transformers, torch
print("sentence-transformers", sentence_transformers.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "Activez le GPU : Settings > Accelerator > GPU T4"

In [ ]:
import os
os.makedirs("/kaggle/working/backend/training", exist_ok=True)
open("/kaggle/working/backend/training/__init__.py", "w").close()
%cd /kaggle/working/backend

## 1. Scripts du projet (copies exactes de `backend/training/`)

In [ ]:
%%writefile training/prepare_data.py
"""Télécharge des jeux de données publics de recherche (FR + EN) et construit :

- data/training/train_pairs.jsonl      : {"query", "positive"}            (négatifs « in-batch »)
- data/training/train_triplets.jsonl   : {"query", "positive", "negative"} (négatifs difficiles)
- data/training/eval_<nom>.json        : {"queries", "corpus", "qrels"}   (jeux de test tenus à l'écart)

Sources (Hugging Face Hub, toutes publiques) :
  etalab-ia/piaf                 QA français natif sur Wikipédia          -> entraînement
  lincoln/newsquadfr             QA français sur articles de presse       -> entraînement + test
  sentence-transformers/miracl   requêtes FR + négatifs difficiles        -> entraînement
  lyon-nlp/alloprof              questions d'élèves -> fiches de cours    -> entraînement + test
  sentence-transformers/squad    QA anglais (maintien du multilingue)     -> entraînement
  manu/fquad2_test               FQuAD (validation)                       -> test uniquement

Usage : python -m training.prepare_data
"""
from __future__ import annotations

import ast
import json
import random
from pathlib import Path

import pandas as pd
from huggingface_hub import hf_hub_download

OUT = Path(__file__).resolve().parents[1] / "data" / "training"
SEED = 42
MAX_PASSAGE_CHARS = 1500  # ~ taille de 1 à 2 chunks DocuMind


def _dl(repo: str, filename: str) -> str:
    return hf_hub_download(repo, filename, repo_type="dataset")


def _clean(text: str) -> str:
    return " ".join(str(text).split())


def _jsonl(path: str) -> list[dict]:
    with open(path, encoding="utf-8") as fh:
        return [json.loads(line) for line in fh if line.strip()]


def _eval_set(queries: dict[str, str], corpus: dict[str, str], qrels: dict[str, list[str]]) -> dict:
    return {"queries": queries, "corpus": corpus, "qrels": qrels}


def main() -> None:
    rng = random.Random(SEED)
    OUT.mkdir(parents=True, exist_ok=True)
    pairs: list[dict] = []
    triplets: list[dict] = []
    stats: dict[str, int] = {}

    # ---- PIAF (entraînement) -------------------------------------------------
    piaf = pd.read_parquet(_dl("etalab-ia/piaf", "plain_text/train-00000-of-00001.parquet"))
    piaf_contexts = sorted({_clean(c) for c in piaf["context"]})
    for q, c in zip(piaf["question"], piaf["context"]):
        pairs.append({"query": _clean(q), "positive": _clean(c)[:MAX_PASSAGE_CHARS], "src": "piaf"})
    stats["piaf"] = len(piaf)

    # ---- newsquadfr (entraînement sur train, test sur valid+test) ------------
    news_train = _jsonl(_dl("lincoln/newsquadfr", "train.json"))
    news_eval = _jsonl(_dl("lincoln/newsquadfr", "valid.json")) + _jsonl(_dl("lincoln/newsquadfr", "test.json"))
    eval_paragraphs = {r["paragraph_id"] for r in news_eval}
    eval_articles = {r["article_id"] for r in news_eval}
    n = 0
    for r in news_train:
        if r["article_id"] in eval_articles:  # pas de fuite d'article entre train et test
            continue
        pairs.append({"query": _clean(r["question"]), "positive": _clean(r["context"]), "src": "newsquadfr"})
        n += 1
    stats["newsquadfr"] = n
    corpus = {f"n{r['paragraph_id']}": _clean(r["context"]) for r in news_train + news_eval}
    queries = {f"q{r['id']}": _clean(r["question"]) for r in news_eval}
    qrels = {f"q{r['id']}": [f"n{r['paragraph_id']}"] for r in news_eval}
    assert all(f"n{p}" in corpus for p in eval_paragraphs)
    (OUT / "eval_newsquadfr.json").write_text(json.dumps(_eval_set(queries, corpus, qrels), ensure_ascii=False), "utf-8")

    # ---- MIRACL fr (triplets avec négatifs difficiles) -----------------------
    miracl = pd.read_parquet(_dl("sentence-transformers/miracl", "fr-triplet/train-00000-of-00001.parquet"))
    for a, p, ng in zip(miracl["anchor"], miracl["positive"], miracl["negative"]):
        triplets.append({"query": _clean(a), "positive": _clean(p)[:MAX_PASSAGE_CHARS],
                         "negative": _clean(ng)[:MAX_PASSAGE_CHARS], "src": "miracl-fr"})
    stats["miracl-fr"] = len(miracl)

    # ---- Alloprof (questions d'élèves -> fiches) ------------------------------
    docs = json.load(open(_dl("lyon-nlp/alloprof", "documents.json"), encoding="utf-8"))
    doc_text = {d["uuid"]: _clean(f"{d['title']}. {d['text']}") for d in docs}
    q_train = json.load(open(_dl("lyon-nlp/alloprof", "queries-train.json"), encoding="utf-8"))
    q_test = json.load(open(_dl("lyon-nlp/alloprof", "queries-test.json"), encoding="utf-8"))

    def _rel(q: dict) -> list[str]:
        rel = q["relevant"]
        if isinstance(rel, str):
            rel = ast.literal_eval(rel)
        return [r for r in rel if r in doc_text]

    rng.shuffle(q_train)
    n = 0
    for q in q_train:
        rel = _rel(q)
        if not rel or len(q["text"]) < 15:
            continue
        pairs.append({"query": _clean(q["text"])[:400], "positive": doc_text[rel[0]][:MAX_PASSAGE_CHARS], "src": "alloprof"})
        n += 1
        if n >= 3000:
            break
    stats["alloprof"] = n
    rng.shuffle(q_test)
    queries, qrels = {}, {}
    for q in q_test:
        rel = _rel(q)
        if rel and len(queries) < 500:
            queries[f"a{q['id']}"] = _clean(q["text"])[:400]
            qrels[f"a{q['id']}"] = rel
    corpus = {k: v[:3000] for k, v in doc_text.items()}
    (OUT / "eval_alloprof.json").write_text(json.dumps(_eval_set(queries, corpus, qrels), ensure_ascii=False), "utf-8")

    # ---- SQuAD EN (échantillon, pour ne pas dégrader l'anglais) ---------------
    squad = pd.read_parquet(_dl("sentence-transformers/squad", "pair/train-00000-of-00001.parquet"))
    squad = squad.sample(n=1500, random_state=SEED)
    for q, c in zip(squad["question"], squad["answer"]):
        pairs.append({"query": _clean(q), "positive": _clean(c)[:MAX_PASSAGE_CHARS], "src": "squad-en"})
    stats["squad-en"] = len(squad)

    # ---- FQuAD (test uniquement ; contextes PIAF ajoutés comme distracteurs) --
    fq = pd.read_parquet(_dl("manu/fquad2_test", "data/valid-00000-of-00001-c57e7fc735be9d5f.parquet"))
    ctx_ids: dict[str, str] = {}
    for c in fq["context"]:
        ctx_ids.setdefault(_clean(c), f"f{len(ctx_ids)}")
    corpus = {cid: c for c, cid in ctx_ids.items()}
    for i, c in enumerate(piaf_contexts):
        corpus.setdefault(f"p{i}", c)
    queries, qrels = {}, {}
    for i, row in enumerate(fq.itertuples()):
        if not row.is_impossible:
            queries[f"fq{i}"] = _clean(row.question)
            qrels[f"fq{i}"] = [ctx_ids[_clean(row.context)]]
    (OUT / "eval_fquad.json").write_text(json.dumps(_eval_set(queries, corpus, qrels), ensure_ascii=False), "utf-8")

    rng.shuffle(pairs)
    rng.shuffle(triplets)
    with open(OUT / "train_pairs.jsonl", "w", encoding="utf-8") as fh:
        for r in pairs:
            fh.write(json.dumps(r, ensure_ascii=False) + "\n")
    with open(OUT / "train_triplets.jsonl", "w", encoding="utf-8") as fh:
        for r in triplets:
            fh.write(json.dumps(r, ensure_ascii=False) + "\n")
    stats["total_pairs"] = len(pairs)
    stats["total_triplets"] = len(triplets)
    (OUT / "stats.json").write_text(json.dumps(stats, indent=2), "utf-8")
    print(json.dumps(stats, indent=2))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile training/train_embeddings.py
"""Affinage (fine-tuning) du modèle d'embeddings de DocuMind pour la recherche documentaire.

Modèle de base : intfloat/multilingual-e5-small (384 dimensions, rapide sur CPU).
Perte : MultipleNegativesRankingLoss (contrastif, négatifs in-batch + négatifs difficiles MIRACL).

Usage :
    python -m training.prepare_data
    python -m training.train_embeddings --max-steps 400
    python -m training.evaluate_retrieval
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path

from datasets import Dataset
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.training_args import BatchSamplers, MultiDatasetBatchSamplers

ROOT = Path(__file__).resolve().parents[1]
DATA = ROOT / "data" / "training"
DEFAULT_OUT = ROOT / "models" / "documind-e5-small-ft"


def _load(path: Path) -> list[dict]:
    with open(path, encoding="utf-8") as fh:
        return [json.loads(line) for line in fh]


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--base-model", default="intfloat/multilingual-e5-small")
    ap.add_argument("--output", default=str(DEFAULT_OUT))
    ap.add_argument("--batch-size", type=int, default=32)
    ap.add_argument("--max-steps", type=int, default=400)
    ap.add_argument("--lr", type=float, default=2e-5)
    ap.add_argument("--max-seq-length", type=int, default=192)
    ap.add_argument("--epochs", type=float, default=None, help="si fourni, remplace --max-steps (entraînement GPU)")
    ap.add_argument("--device", default="auto", help="auto | cpu | cuda")
    args = ap.parse_args()

    import torch

    device = ("cuda" if torch.cuda.is_available() else "cpu") if args.device == "auto" else args.device
    print("Périphérique :", device)
    model = SentenceTransformer(args.base_model, device=device)
    model.max_seq_length = args.max_seq_length

    # Les modèles E5 attendent les préfixes « query: » / « passage: ».
    pairs = _load(DATA / "train_pairs.jsonl")
    triplets = _load(DATA / "train_triplets.jsonl")
    ds_pairs = Dataset.from_dict({
        "anchor": ["query: " + r["query"] for r in pairs],
        "positive": ["passage: " + r["positive"] for r in pairs],
    })
    ds_triplets = Dataset.from_dict({
        "anchor": ["query: " + r["query"] for r in triplets],
        "positive": ["passage: " + r["positive"] for r in triplets],
        "negative": ["passage: " + r["negative"] for r in triplets],
    })
    print(f"paires={len(ds_pairs)} triplets={len(ds_triplets)}")

    loss = MultipleNegativesRankingLoss(model, scale=50.0)  # température 0.02
    targs = SentenceTransformerTrainingArguments(
        output_dir=str(ROOT / "data" / "training" / "checkpoints"),
        max_steps=-1 if args.epochs else args.max_steps,
        num_train_epochs=args.epochs or 1,
        fp16=device == "cuda",
        per_device_train_batch_size=args.batch_size,
        learning_rate=args.lr,
        warmup_ratio=0.1,
        weight_decay=0.01,
        batch_sampler=BatchSamplers.NO_DUPLICATES,
        multi_dataset_batch_sampler=MultiDatasetBatchSamplers.PROPORTIONAL,
        logging_steps=20,
        save_strategy="no",
        report_to="none",
        use_cpu=device == "cpu",
        dataloader_num_workers=0,
        seed=42,
    )
    trainer = SentenceTransformerTrainer(
        model=model,
        args=targs,
        train_dataset={"pairs": ds_pairs, "triplets": ds_triplets},
        loss=loss,
    )
    trainer.train()
    model.max_seq_length = 512
    model.save(args.output)
    (Path(args.output) / "training_info.json").write_text(json.dumps({
        "base_model": args.base_model, "max_steps": trainer.state.global_step, "epochs": args.epochs,
        "device": device, "batch_size": args.batch_size,
        "lr": args.lr, "train_max_seq_length": args.max_seq_length,
        "n_pairs": len(ds_pairs), "n_triplets": len(ds_triplets),
        "log": [h for h in trainer.state.log_history if "loss" in h],
    }, indent=2), "utf-8")
    print("Modèle enregistré :", args.output)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile training/evaluate_retrieval.py
"""Compare des modèles d'embeddings sur les jeux de test tenus à l'écart (FQuAD, newsquadfr, Alloprof).

Métriques : Recall@1/5/10, MRR@10, nDCG@10. Résultat écrit dans reports/retrieval_benchmark.json.

Usage : python -m training.evaluate_retrieval [--models chemin_ou_nom ...]
"""
from __future__ import annotations

import argparse
import json
import math
import time
from pathlib import Path

import numpy as np
from sentence_transformers import SentenceTransformer

ROOT = Path(__file__).resolve().parents[1]
DATA = ROOT / "data" / "training"
REPORT = ROOT / "reports" / "retrieval_benchmark.json"


def metrics(ranked: list[list[str]], gold: list[set[str]]) -> dict[str, float]:
    out = {"recall@1": 0.0, "recall@5": 0.0, "recall@10": 0.0, "mrr@10": 0.0, "ndcg@10": 0.0}
    for ids, rel in zip(ranked, gold):
        for k in (1, 5, 10):
            out[f"recall@{k}"] += float(bool(rel & set(ids[:k])))
        for rank, i in enumerate(ids[:10], 1):
            if i in rel:
                out["mrr@10"] += 1.0 / rank
                break
        dcg = sum(1.0 / math.log2(r + 1) for r, i in enumerate(ids[:10], 1) if i in rel)
        idcg = sum(1.0 / math.log2(r + 1) for r in range(1, min(len(rel), 10) + 1))
        out["ndcg@10"] += dcg / idcg
    return {k: round(v / len(ranked), 4) for k, v in out.items()}


def evaluate(model_name: str, sets: dict[str, dict], batch_size: int = 32) -> dict:
    import torch

    model = SentenceTransformer(model_name, device="cuda" if torch.cuda.is_available() else "cpu")
    model.max_seq_length = 512
    res = {}
    for name, data in sets.items():
        t0 = time.time()
        cids = list(data["corpus"])
        qids = list(data["queries"])
        c = model.encode(["passage: " + data["corpus"][i] for i in cids], batch_size=batch_size,
                         normalize_embeddings=True, show_progress_bar=False)
        q = model.encode(["query: " + data["queries"][i] for i in qids], batch_size=batch_size,
                         normalize_embeddings=True, show_progress_bar=False)
        top = np.argsort(-(q @ c.T), axis=1)[:, :10]
        ranked = [[cids[j] for j in row] for row in top]
        gold = [set(data["qrels"][i]) for i in qids]
        res[name] = metrics(ranked, gold) | {"n_queries": len(qids), "n_corpus": len(cids),
                                             "seconds": round(time.time() - t0, 1)}
        print(model_name, name, res[name], flush=True)
    res["mean_ndcg@10"] = round(float(np.mean([v["ndcg@10"] for v in res.values()])), 4)
    res["mean_recall@5"] = round(float(np.mean([v["recall@5"] for k, v in res.items() if isinstance(v, dict)])), 4)
    return res


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--models", nargs="+", default=["intfloat/multilingual-e5-small",
                                                    str(ROOT / "models" / "documind-e5-small-ft")])
    ap.add_argument("--batch-size", type=int, default=32)
    args = ap.parse_args()
    sets = {p.stem.replace("eval_", ""): json.loads(p.read_text("utf-8")) for p in sorted(DATA.glob("eval_*.json"))}
    report = {m: evaluate(m, sets, args.batch_size) for m in args.models}
    REPORT.parent.mkdir(parents=True, exist_ok=True)
    existing = json.loads(REPORT.read_text("utf-8")) if REPORT.exists() else {}
    existing.update(report)
    REPORT.write_text(json.dumps(existing, indent=2, ensure_ascii=False), "utf-8")
    print(json.dumps({m: {"mean_ndcg@10": r["mean_ndcg@10"], "mean_recall@5": r["mean_recall@5"]}
                      for m, r in existing.items()}, indent=2))


if __name__ == "__main__":
    main()

## 2. Téléchargement et préparation des données (Hugging Face)

In [ ]:
!python -m training.prepare_data

## 3. Entraînement sur GPU
Un seul GPU est utilisé (`CUDA_VISIBLE_DEVICES=0`) pour éviter le mode multi-GPU.

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python -m training.train_embeddings --epochs 3 --batch-size 64 --max-seq-length 256 --lr 2e-5

## 4. Benchmark sur les jeux de test tenus à l'écart : modèle de base contre modèle affiné

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python -m training.evaluate_retrieval --batch-size 128

In [ ]:
import json
rep = json.load(open("reports/retrieval_benchmark.json", encoding="utf-8"))
for model, r in rep.items():
    print()
    print("===", model)
    for k, v in r.items():
        if isinstance(v, dict):
            print(f"  {k:12s} R@1={v['recall@1']:.3f}  R@5={v['recall@5']:.3f}  R@10={v['recall@10']:.3f}  MRR@10={v['mrr@10']:.3f}  nDCG@10={v['ndcg@10']:.3f}  (n={v['n_queries']})")
    print(f"  MOYENNE      R@5={r['mean_recall@5']:.3f}  nDCG@10={r['mean_ndcg@10']:.3f}")
print()
print("Journal d'entraînement :")
info = json.load(open("models/documind-e5-small-ft/training_info.json"))
print({k: v for k, v in info.items() if k != "log"})
for h in info["log"]:
    print(f"  pas {h['step']:4d}  perte {h['loss']:.4f}")


## 5. Archive à télécharger

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/documind-e5-small-ft", "zip", "models/documind-e5-small-ft")
shutil.copy("reports/retrieval_benchmark.json", "/kaggle/working/retrieval_benchmark.json")
!ls -lh /kaggle/working/*.zip /kaggle/working/*.json